In [0]:
from pyspark.sql.functions import col, current_timestamp, current_date


catalog_name = 'dbr_dev_ua5816bd'
login = 'daniloshurko'
target_table = f"{catalog_name}.{login}_bronze.games"
file_path = "/Volumes/dbr_dev_ua5816bd/daniloshurko/raw_data/games.csv"

df_raw = (spark.read
          .format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(file_path))

clean_columns = [c.strip().replace(" ", "_") for c in df_raw.columns]
df_raw = df_raw.toDF(*clean_columns)

df_bronze = (df_raw
             .dropDuplicates(["AppID"])
             .withColumn("source_filename", col("_metadata.file_name"))
             .withColumn("ingestion_timestamp", current_timestamp())
             .withColumn("load_date", current_date()))

(df_bronze.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(target_table))

print(f"Data successfully ingested into {target_table}")